# AC-OPF dataset condensation with gradient matching

This small example alternates **matching gradients → feasible input update → detached student SGD**.
It uses one architecture per experiment and solver labels with stopped gradients.
The [README](README.md) explains the physical constraints and differences from Zhao et al.
The distributed runner adds fresh student replicas, periodic restarts, evaluation and resume support.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lumina').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import torch
import pandas as pd
from torch_geometric.data import Batch
from threadpoolctl import threadpool_limits
from distillation.synthetic import generate, describe
from distillation.synthetic.experiments import balanced_sizes, case_bus_range
from distillation.synthetic.learning import benchmark_graph, model_for, model_options
from distillation.DC.dcgm import SyntheticInputs, data_gradients, feasible_step, train_student

torch.set_num_threads(1)
thread_limits = threadpool_limits(limits=1)
CASE, MODEL, PROTOCOL = 30, 'GAT', 'random_size'
DEVICE = torch.device('xpu:0' if torch.xpu.is_available() else 'cpu')
ARTIFACT_ROOT = Path('/lus/flare/projects/FRAME-IDP/siebenschuh/acopf_datasets/dataset_distillation')
REFERENCE_ROOT = ARTIFACT_ROOT / 'campaign_20261010/experiments/GAT'
OUTPUT_ROOT = ARTIFACT_ROOT / 'DCGM/mvp'
config = dict(case=CASE, model=MODEL, model_options=model_options(MODEL), model_precision="float64", bus_range=case_bus_range(CASE))
config


## Training-only references and fresh feasible inputs

The saved reference file contains training examples. Generation uses the first twelve;
the matching batch uses a separate eight. These small demo batches are not an evaluation set.
Random-size inputs span ±25% and keep the total number of buses equal to six base cases.


In [ ]:
reference = torch.load(REFERENCE_ROOT / f'case{CASE}' / 'original.pt', weights_only=False, map_location='cpu')
references, real_graphs = reference['graphs'][:12], reference['graphs'][12:20]
mode = 'conservative' if PROTOCOL == 'conservative' else 'random'
sizes = balanced_sizes(CASE, 1, 6) if PROTOCOL == 'random_size' else [CASE] * 6
result = generate(references, mode=mode, count=6, original_buses=CASE,
                  bus_range=config['bus_range'], sizes=None if mode == 'conservative' else sizes,
                  reference_ids=reference['ids'][:12], seed=42, attempts=64)
synthetic = SyntheticInputs(result.physical_graphs, strength=.02)
real = Batch.from_data_list([benchmark_graph(g, labels=True) for g in real_graphs])
display(pd.DataFrame([describe(g) for g in result.physical_graphs]))


## Two matching rounds and an intervening student epoch

Synthetic SGD uses momentum 0.5. The model uses ordinary SGD on detached inputs;
its learning rate is deliberately small for this demonstration. Campaign runs calibrate a
fixed rate on training data. The last matching round is followed by a fresh initialization,
matching the reference algorithm's restart convention.


In [ ]:
model = model_for(references[0], config, seed=100, device=DEVICE)
data_optimizer = torch.optim.SGD(synthetic.parameters(), lr=.1, momentum=.5)
student_optimizer = torch.optim.SGD(model.parameters(), lr=1e-6)
physical = result.physical_graphs
history = []
for round_index in range(2):
    data_optimizer.zero_grad(set_to_none=True)
    value, diagnostics = data_gradients(model, synthetic, real)
    accepted, proposed, checks, attempts = feasible_step(synthetic, data_optimizer)
    if accepted: physical = proposed
    row = dict(round=round_index+1, matching_loss=value, accepted=accepted,
               accepted_fraction=attempts[-1]['fraction'] if accepted else 0., **diagnostics)
    if round_index == 0:
        row['student_mse'] = train_student(model, student_optimizer, synthetic,
                                            epochs=1, batch_size=3, seed=200)
    history.append(row)
display(pd.DataFrame(history))
restarted_model = model_for(references[0], config, seed=101, device=DEVICE)


## Export inputs separately from supervision and certificates

All exported labels come from successful AC-OPF solves. A lower matching loss alone
does not establish usefulness: the campaign evaluates fresh students against random-real
and initial-synthetic controls and reports prediction errors and physical residuals.


In [ ]:
inputs, labels = synthetic.batch(detached=True)
output = OUTPUT_ROOT / MODEL / f'case{CASE}' / PROTOCOL / 'mvp.pt'
output.parent.mkdir(parents=True, exist_ok=True)
torch.save(dict(format='dcgm-demo-v1', config=config, inputs=inputs,
                teaching_targets=labels, physical_solved_graphs=physical,
                generation=result.metadata, history=history,
                reference_ids=reference['ids'][:12], matching_ids=reference['ids'][12:20],
                synthetic_state=synthetic.state_dict(), optimizer=data_optimizer.state_dict(),
                student={k: v.detach().cpu() for k,v in model.state_dict().items()}), output)
display({'artifact': str(output), 'device': str(DEVICE), 'graphs': inputs.num_graphs,
         'bus_range': config['bus_range'], 'detached_targets': all(not t.requires_grad for t in labels.values())})
